In [ ]:
# SET UP COMUNE

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Estrai preprocessing e modello addestrato dalla tua pipeline
ct_fitted = best_pipeline.named_steps['preprocessing']
model = best_pipeline.named_steps['model']

feature_names = ct_fitted.get_feature_names_out()
X_train_df = pd.DataFrame(ct_fitted.transform(X_train), columns=feature_names)
X_test_df = pd.DataFrame(ct_fitted.transform(X_test), columns=feature_names)

In [ ]:
# SHAP

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import shap
from xgboost import XGBClassifier

# 1. Selezione automatica dell'Explainer in base al tipo di modello
if isinstance(model, (RandomForestClassifier, XGBClassifier)):
  explainer = shap.TreeExplainer(model)
  shap_values = explainer(X_test_df)
  # Per RandomForest in sklearn, shap_values ha shape (n_samples, n_features, 2) -> prendiamo la classe 1
  sv = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values

elif isinstance(model, (LogisticRegression, LinearSVC)):
  explainer = shap.LinearExplainer(model, X_train_df)
  sv = explainer(X_test_df)

else:
  # Per KNeighborsClassifier e SVC(kernel='rbf', probability=True)
  # Usiamo un background riassuntivo (es. kmeans o tutto X_train_df visto che sono ~90 pazienti)
  explainer = shap.KernelExplainer(
      lambda x: model.predict_proba(x)[:, 1], X_train_df
  )
  shap_vals_array = explainer.shap_values(X_test_df)
  sv = shap.Explanation(
      values=shap_vals_array,
      base_values=explainer.expected_value,
      data=X_test_df.values,
      feature_names=feature_names,
  )

# 2. Plot Globale (Beeswarm: importanza + direzione dell'effetto)
plt.figure()
shap.plots.beeswarm(sv, max_display=15)

# 3. Plot Locale per un singolo paziente del test set (es. paziente indice 0)
plt.figure()
shap.plots.waterfall(sv[0], max_display=12)

In [ ]:
# LIME 

import lime.lime_tabular

# Inizializza l'explainer tabellare sui dati di training trasformati
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_df.values,
    feature_names=list(feature_names),
    class_names=['Eosinofili <= 0.8', 'Eosinofili > 0.8'],
    mode='classification',
    random_state=42,
)

# Funzione di predizione (gestisce anche LinearSVC che non ha predict_proba nativo)
if hasattr(model, 'predict_proba'):
  predict_fn = model.predict_proba
else:
  # Per LinearSVC: converte decision_function in pseudo-probabilità tramite sigmoide
  from scipy.special import expit

  def predict_fn(x):
    probs_1 = expit(model.decision_function(x))
    return np.column_stack([1 - probs_1, probs_1])


# Spiega un singolo paziente del test set (es. paziente 0)
patient_idx = 0
exp = lime_explainer.explain_instance(
    data_row=X_test_df.iloc[patient_idx].values,
    predict_fn=predict_fn,
    num_features=12,
)

# Visualizza grafico matplotlib
fig = exp.as_pyplot_figure()
fig.set_size_inches(7, 4.5)
plt.title(f'LIME Explanation - Paziente Test #{patient_idx}')
plt.tight_layout()
plt.show()

In [ ]:
# PARTIAL DEPENDENCE 

import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

# Scegli le feature più importanti (es. le top 4 emerse da SHAP o Permutation Importance)
# Puoi passare i nomi esatti presenti in feature_names
features_to_plot = list(perm_df['Feature'].head(4))

# Nota: per LinearSVC che non ha predict_proba, PartialDependenceDisplay usa in automatico
# response_method="auto" (che usa predict_proba se disponibile, altrimenti decision_function)
fig, ax = plt.subplots(figsize=(9, 6), layout='constrained')

PartialDependenceDisplay.from_estimator(
    estimator=model,
    X=X_test_df,
    features=features_to_plot,
    kind='both',  # 'both' mostra sia la media globale (PDP) che i singoli pazienti (ICE)
    subsample=50,  # Numero max di linee ICE da disegnare
    n_cols=2,  # Griglia 2x2 per i 4 grafici
    grid_resolution=30,  # Risoluzione dell'asse X
    random_state=42,
    ax=ax,
    ice_lines_kw={'color': '#5bc0de', 'alpha': 0.25, 'linewidth': 0.8},
    pd_line_kw={
        'color': '#d9534f',
        'linewidth': 2.2,
        'label': 'Media (PDP)',
    },
)

fig.suptitle(
    'Partial Dependence & ICE Plots (Top 4 Features)', fontsize=11, y=1.02
)
plt.show()